# NB07 · Robustez ante perturbaciones de imagen

**Entorno recomendado:** GPU T4 o L4  
**Tiempo estimado:** GPU, 8–15 min; CPU, 25–40 min

Este notebook evalúa el modelo final `YOLO11m-cls · semilla 2026` sobre las **70 imágenes del conjunto de prueba**. Cada fotografía se compara consigo misma en condición original y bajo alteraciones controladas de:

1. oscurecimiento;
2. sobreexposición;
3. reducción de contraste;
4. desenfoque gaussiano;
5. ruido aditivo;
6. rotación;
7. compresión JPEG.

Se calculan exactitud, exactitud balanceada, F1 macro, estabilidad de etiqueta y pérdida de F1 macro frente a la condición original. La comparación es pareada: todas las condiciones contienen las mismas imágenes.

> Las perturbaciones son simulaciones controladas y no sustituyen una validación externa con fotografías nuevas de campo. El conjunto de prueba no se utiliza para reentrenar ni seleccionar el modelo.


In [ ]:
%pip install -q -U ultralytics


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime
from io import BytesIO
import json
import math
import platform
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import ultralytics
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
)
from ultralytics import YOLO

PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
DRIVE_DATA = PROJECT_ROOT / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
MODEL_PATH = PROJECT_ROOT / '05_MODELOS' / 'yolo11m_cls_seed2026' / 'best.pt'
RESULTS_ROOT = PROJECT_ROOT / '06_RESULTADOS' / 'NB07_ROBUSTEZ_PERTURBACIONES'
BOOK_ROOT = PROJECT_ROOT / '08_MATERIAL_LIBRO' / 'NB07_ROBUSTEZ_PERTURBACIONES'

LOCAL_DATA = Path('/content/PALMA_CLS_NB07')
LOCAL_CACHE = Path('/content/PALMA_ROBUSTEZ_CACHE')
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_RESULTS = RESULTS_ROOT / RUN_ID
RUN_BOOK = BOOK_ROOT / RUN_ID
FIGURES_DIR = RUN_BOOK / 'figuras'
TABLES_DIR = RUN_BOOK / 'tablas'

for path in [RUN_RESULTS, FIGURES_DIR, TABLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not DRIVE_DATA.exists():
    raise FileNotFoundError(f'No se encontró el conjunto procesado: {DRIVE_DATA}. Ejecute NB02.')
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'No se encontró el modelo final: {MODEL_PATH}. Ejecute NB04.')

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
DEVICE_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
IMGSZ = 384
BATCH = 32 if torch.cuda.is_available() else 8
SEED = 2026
BOOTSTRAP_REPETITIONS = 2000
CLASSES = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']
SPANISH = {
    'damaged': 'dañado',
    'empty': 'vacío',
    'overripe': 'sobremaduro',
    'ripe': 'maduro',
    'unripe': 'inmaduro',
}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

print('Dispositivo:', DEVICE_NAME)
print('Modelo:', MODEL_PATH)
print('Datos:', DRIVE_DATA)
print('Resultados:', RUN_RESULTS)
print('Material para el libro:', RUN_BOOK)
if DEVICE == 'cpu':
    print('ADVERTENCIA: se recomienda activar una GPU T4 o L4 en Colab.')


## 1. Preparación del conjunto de prueba

Se copia temporalmente el conjunto procesado desde Drive al almacenamiento local de Colab. Esto acelera la creación de perturbaciones y la inferencia. La copia temporal se elimina al finalizar el entorno de ejecución.


In [ ]:
for temporary_path in [LOCAL_DATA, LOCAL_CACHE]:
    if temporary_path.exists():
        shutil.rmtree(temporary_path)

shutil.copytree(DRIVE_DATA, LOCAL_DATA)
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

test_paths = sorted(
    p for p in (LOCAL_DATA / 'test').glob('*/*')
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)
if len(test_paths) != 70:
    raise RuntimeError(f'Se esperaban 70 imágenes de prueba y se encontraron {len(test_paths)}.')

test_manifest = pd.DataFrame({
    'image_id': [str(p.relative_to(LOCAL_DATA / 'test')) for p in test_paths],
    'true_class': [p.parent.name for p in test_paths],
    'source_path': [str(p) for p in test_paths],
})
if not set(test_manifest['true_class']).issubset(CLASSES):
    raise RuntimeError('Se encontraron clases inesperadas en el conjunto de prueba.')

display(test_manifest.groupby('true_class').size().rename('n').reset_index())
print('Total de imágenes de prueba:', len(test_manifest))


## 2. Diseño de perturbaciones

Cada familia tiene dos intensidades: **moderada** y **severa**. Los parámetros se fijan antes de observar los resultados. La condición `Original` funciona como referencia.

La rotación rellena los bordes con el color mediano de cada imagen para evitar que triángulos negros artificiales se conviertan en una señal para el modelo. El ruido se genera con semilla fija, por lo que el experimento es reproducible.


In [ ]:
CONDITIONS = [
    {'id': 'clean', 'family': 'Original', 'level': 0, 'level_name': 'Referencia', 'parameter': 0, 'label': 'Original'},
    {'id': 'dark_moderate', 'family': 'Oscurecimiento', 'level': 1, 'level_name': 'Moderado', 'parameter': 0.75, 'label': 'Oscurecimiento · moderado'},
    {'id': 'dark_severe', 'family': 'Oscurecimiento', 'level': 2, 'level_name': 'Severo', 'parameter': 0.55, 'label': 'Oscurecimiento · severo'},
    {'id': 'bright_moderate', 'family': 'Sobreexposición', 'level': 1, 'level_name': 'Moderado', 'parameter': 1.25, 'label': 'Sobreexposición · moderada'},
    {'id': 'bright_severe', 'family': 'Sobreexposición', 'level': 2, 'level_name': 'Severo', 'parameter': 1.45, 'label': 'Sobreexposición · severa'},
    {'id': 'contrast_moderate', 'family': 'Contraste bajo', 'level': 1, 'level_name': 'Moderado', 'parameter': 0.70, 'label': 'Contraste bajo · moderado'},
    {'id': 'contrast_severe', 'family': 'Contraste bajo', 'level': 2, 'level_name': 'Severo', 'parameter': 0.45, 'label': 'Contraste bajo · severo'},
    {'id': 'blur_moderate', 'family': 'Desenfoque', 'level': 1, 'level_name': 'Moderado', 'parameter': 1.0, 'label': 'Desenfoque · moderado'},
    {'id': 'blur_severe', 'family': 'Desenfoque', 'level': 2, 'level_name': 'Severo', 'parameter': 2.5, 'label': 'Desenfoque · severo'},
    {'id': 'noise_moderate', 'family': 'Ruido', 'level': 1, 'level_name': 'Moderado', 'parameter': 12, 'label': 'Ruido · moderado'},
    {'id': 'noise_severe', 'family': 'Ruido', 'level': 2, 'level_name': 'Severo', 'parameter': 28, 'label': 'Ruido · severo'},
    {'id': 'rotation_moderate', 'family': 'Rotación', 'level': 1, 'level_name': 'Moderado', 'parameter': 8, 'label': 'Rotación · 8°'},
    {'id': 'rotation_severe', 'family': 'Rotación', 'level': 2, 'level_name': 'Severo', 'parameter': 18, 'label': 'Rotación · 18°'},
    {'id': 'jpeg_moderate', 'family': 'Compresión JPEG', 'level': 1, 'level_name': 'Moderado', 'parameter': 60, 'label': 'JPEG · calidad 60'},
    {'id': 'jpeg_severe', 'family': 'Compresión JPEG', 'level': 2, 'level_name': 'Severo', 'parameter': 30, 'label': 'JPEG · calidad 30'},
]
conditions = pd.DataFrame(CONDITIONS)
display(conditions)
print('Condiciones totales:', len(conditions))
print('Inferencias previstas:', len(conditions) * len(test_paths))


In [ ]:
def perturb_image(image, condition_id, parameter, seed):
    image = ImageOps.exif_transpose(image).convert('RGB')

    if condition_id == 'clean':
        return image
    if condition_id.startswith('dark_') or condition_id.startswith('bright_'):
        return ImageEnhance.Brightness(image).enhance(float(parameter))
    if condition_id.startswith('contrast_'):
        return ImageEnhance.Contrast(image).enhance(float(parameter))
    if condition_id.startswith('blur_'):
        return image.filter(ImageFilter.GaussianBlur(radius=float(parameter)))
    if condition_id.startswith('noise_'):
        array = np.asarray(image, dtype=np.float32)
        rng = np.random.default_rng(seed)
        noisy = np.clip(array + rng.normal(0, float(parameter), array.shape), 0, 255)
        return Image.fromarray(noisy.astype(np.uint8), mode='RGB')
    if condition_id.startswith('rotation_'):
        array = np.asarray(image)
        fill = tuple(np.median(array.reshape(-1, 3), axis=0).astype(np.uint8).tolist())
        return image.rotate(
            float(parameter),
            resample=Image.Resampling.BICUBIC,
            expand=False,
            fillcolor=fill,
        )
    if condition_id.startswith('jpeg_'):
        buffer = BytesIO()
        image.save(buffer, format='JPEG', quality=int(parameter), optimize=True)
        buffer.seek(0)
        with Image.open(buffer) as compressed:
            return compressed.convert('RGB').copy()
    raise ValueError(f'Condición desconocida: {condition_id}')


paths_by_condition = {'clean': test_paths}
generation_start = time.perf_counter()

for condition_index, condition in conditions.iloc[1:].iterrows():
    generated_paths = []
    condition_dir = LOCAL_CACHE / condition['id']
    for image_index, source_path in enumerate(test_paths):
        destination_dir = condition_dir / source_path.parent.name
        destination_dir.mkdir(parents=True, exist_ok=True)
        destination = destination_dir / f'{image_index:03d}_{source_path.stem}.jpg'
        with Image.open(source_path) as image:
            transformed = perturb_image(
                image,
                condition['id'],
                condition['parameter'],
                SEED + condition_index * 10000 + image_index,
            )
        save_quality = int(condition['parameter']) if condition['id'].startswith('jpeg_') else 95
        transformed.save(destination, format='JPEG', quality=save_quality, subsampling=0)
        generated_paths.append(destination)
    paths_by_condition[condition['id']] = generated_paths

print(f'Perturbaciones creadas en {time.perf_counter() - generation_start:.1f} s')
print('Archivos temporales:', sum(len(v) for v in paths_by_condition.values()))


## 3. Inferencia pareada

El mismo modelo y la misma configuración de inferencia se aplican a todas las condiciones. Se guardan los puntajes de las cinco clases para permitir auditorías posteriores.


In [ ]:
model = YOLO(str(MODEL_PATH))
model_names = [model.names[i] for i in sorted(model.names)]
if model_names != CLASSES:
    raise RuntimeError(f'Clases inesperadas en el modelo: {model_names}')

prediction_records = []
inference_start = time.perf_counter()

for _, condition in conditions.iterrows():
    condition_id = condition['id']
    condition_paths = paths_by_condition[condition_id]
    results = model.predict(
        source=[str(path) for path in condition_paths],
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        verbose=False,
    )
    if len(results) != len(test_manifest):
        raise RuntimeError(f'Número inesperado de predicciones en {condition_id}.')

    for image_row, result in zip(test_manifest.itertuples(index=False), results):
        scores = result.probs.data.detach().cpu().numpy().astype(float)
        predicted_id = int(scores.argmax())
        record = {
            'image_id': image_row.image_id,
            'true_class': image_row.true_class,
            'condition_id': condition_id,
            'condition_label': condition['label'],
            'family': condition['family'],
            'level': int(condition['level']),
            'level_name': condition['level_name'],
            'parameter': float(condition['parameter']),
            'predicted_class': CLASSES[predicted_id],
            'confidence': float(scores[predicted_id]),
            'correct': bool(CLASSES[predicted_id] == image_row.true_class),
        }
        for class_id, class_name in enumerate(CLASSES):
            record[f'score_{class_name}'] = float(scores[class_id])
        prediction_records.append(record)
    print('Finalizada:', condition['label'])

predictions = pd.DataFrame(prediction_records)
predictions.to_csv(RUN_RESULTS / 'predicciones_todas_las_condiciones.csv', index=False)
print(f'Inferencia completada en {time.perf_counter() - inference_start:.1f} s')
display(predictions.head())


## 4. Métricas y estabilidad de la predicción

La estabilidad indica el porcentaje de imágenes cuya etiqueta predicha permanece igual a la obtenida con la imagen original. Una etiqueta estable no implica necesariamente una predicción correcta; por ello se informa junto con F1 macro y exactitud balanceada.


In [ ]:
clean_predictions = (
    predictions.query("condition_id == 'clean'")
    .set_index('image_id')
    .sort_index()
)
clean_labels = clean_predictions['predicted_class']

metric_rows = []
per_class_rows = []

for _, condition in conditions.iterrows():
    frame = (
        predictions[predictions['condition_id'] == condition['id']]
        .set_index('image_id')
        .loc[clean_predictions.index]
    )
    y_true = frame['true_class'].to_numpy()
    y_pred = frame['predicted_class'].to_numpy()
    recalls = recall_score(y_true, y_pred, labels=CLASSES, average=None, zero_division=0)
    metric_rows.append({
        **condition.to_dict(),
        'n': len(frame),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0),
        'mean_confidence': frame['confidence'].mean(),
        'stable_label_rate': (frame['predicted_class'] == clean_labels).mean(),
        'flip_rate': (frame['predicted_class'] != clean_labels).mean(),
    })
    for class_name, class_recall in zip(CLASSES, recalls):
        per_class_rows.append({
            'condition_id': condition['id'],
            'condition_label': condition['label'],
            'class': class_name,
            'class_es': SPANISH[class_name],
            'recall': float(class_recall),
        })

metrics = pd.DataFrame(metric_rows)
clean_f1 = float(metrics.loc[metrics['id'] == 'clean', 'macro_f1'].iloc[0])
metrics['macro_f1_drop'] = clean_f1 - metrics['macro_f1']
per_class = pd.DataFrame(per_class_rows)

metrics.to_csv(RUN_RESULTS / 'metricas_robustez.csv', index=False)
per_class.to_csv(RUN_RESULTS / 'recall_por_clase_y_condicion.csv', index=False)
metrics.to_csv(TABLES_DIR / 'tabla_metricas_robustez.csv', index=False)

display(metrics[[
    'label', 'accuracy', 'balanced_accuracy', 'macro_f1',
    'macro_f1_drop', 'stable_label_rate', 'mean_confidence'
]].round(4))


## 5. Comparación estadística pareada

Para cada perturbación se aplica la prueba exacta de McNemar sobre los aciertos y errores pareados respecto de la imagen original. Los valores *p* se corrigen con Holm. Además, un bootstrap pareado de 2.000 repeticiones estima el intervalo de confianza del 95 % para la pérdida de F1 macro.

La prueba indica si cambió la proporción de aciertos; el tamaño del efecto se interpreta mediante la pérdida de F1 macro y su intervalo de confianza.


In [ ]:
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted_sorted = np.empty_like(p_values)
    running_max = 0.0
    m = len(p_values)
    for rank, original_index in enumerate(order):
        candidate = min(1.0, (m - rank) * p_values[original_index])
        running_max = max(running_max, candidate)
        adjusted_sorted[original_index] = running_max
    return adjusted_sorted


baseline_true = clean_predictions['true_class'].to_numpy()
baseline_pred = clean_predictions['predicted_class'].to_numpy()
baseline_correct = baseline_pred == baseline_true

rng = np.random.default_rng(SEED)
bootstrap_indices = rng.integers(
    0,
    len(clean_predictions),
    size=(BOOTSTRAP_REPETITIONS, len(clean_predictions)),
)

statistics_rows = []
for _, condition in conditions.query("id != 'clean'").iterrows():
    frame = (
        predictions[predictions['condition_id'] == condition['id']]
        .set_index('image_id')
        .loc[clean_predictions.index]
    )
    perturbed_pred = frame['predicted_class'].to_numpy()
    perturbed_correct = perturbed_pred == baseline_true
    lost = int(np.sum(baseline_correct & ~perturbed_correct))
    gained = int(np.sum(~baseline_correct & perturbed_correct))
    discordant = lost + gained
    mcnemar_p = (
        float(binomtest(min(lost, gained), discordant, 0.5, alternative='two-sided').pvalue)
        if discordant > 0 else 1.0
    )

    drops = np.empty(BOOTSTRAP_REPETITIONS, dtype=float)
    for bootstrap_id, sampled in enumerate(bootstrap_indices):
        sampled_true = baseline_true[sampled]
        clean_sample = baseline_pred[sampled]
        perturbed_sample = perturbed_pred[sampled]
        clean_sample_f1 = f1_score(
            sampled_true, clean_sample, labels=CLASSES, average='macro', zero_division=0
        )
        perturbed_sample_f1 = f1_score(
            sampled_true, perturbed_sample, labels=CLASSES, average='macro', zero_division=0
        )
        drops[bootstrap_id] = clean_sample_f1 - perturbed_sample_f1

    statistics_rows.append({
        'condition_id': condition['id'],
        'condition_label': condition['label'],
        'lost_correct_predictions': lost,
        'gained_correct_predictions': gained,
        'discordant_pairs': discordant,
        'mcnemar_p_raw': mcnemar_p,
        'macro_f1_drop': clean_f1 - f1_score(
            baseline_true, perturbed_pred, labels=CLASSES, average='macro', zero_division=0
        ),
        'macro_f1_drop_ci95_low': float(np.quantile(drops, 0.025)),
        'macro_f1_drop_ci95_high': float(np.quantile(drops, 0.975)),
    })

statistics = pd.DataFrame(statistics_rows)
statistics['mcnemar_p_holm'] = holm_adjust(statistics['mcnemar_p_raw'])
statistics['mcnemar_significant_holm_005'] = statistics['mcnemar_p_holm'] < 0.05
statistics.to_csv(RUN_RESULTS / 'comparaciones_pareadas.csv', index=False)
statistics.to_csv(TABLES_DIR / 'tabla_comparaciones_pareadas.csv', index=False)
display(statistics.round(4))


## 6. Figuras para el libro

Las figuras se exportan en PNG a 300 dpi. La primera resume la pérdida de F1 macro con incertidumbre; la segunda muestra las curvas de degradación; la tercera permite observar visualmente las alteraciones; y la cuarta presenta la matriz de confusión bajo la condición más exigente para el modelo.


In [ ]:
sns.set_theme(style='whitegrid', context='notebook')

# Figura 1. Pérdida de F1 macro e intervalo de confianza pareado.
plot_stats = statistics.sort_values('macro_f1_drop', ascending=True).reset_index(drop=True)
y = np.arange(len(plot_stats))
x = plot_stats['macro_f1_drop'].to_numpy()
xerr = np.vstack([
    np.maximum(0, x - plot_stats['macro_f1_drop_ci95_low'].to_numpy()),
    np.maximum(0, plot_stats['macro_f1_drop_ci95_high'].to_numpy() - x),
])
colors = np.where(x > 0, '#c0392b', '#1b7f5a')

fig, ax = plt.subplots(figsize=(9.2, 7.6), constrained_layout=True)
ax.barh(y, x, color=colors, alpha=0.88, edgecolor='white')
ax.errorbar(x, y, xerr=xerr, fmt='none', ecolor='#263238', capsize=3, linewidth=1.1)
ax.axvline(0, color='#263238', linewidth=1)
ax.set_yticks(y, plot_stats['condition_label'])
ax.set_xlabel('Pérdida de F1 macro respecto de la imagen original')
ax.set_title('Sensibilidad del clasificador a perturbaciones controladas', fontweight='bold')
ax.grid(axis='x', alpha=0.25)
ax.grid(axis='y', visible=False)
for yi, value in zip(y, x):
    ax.text(value, yi, f'  {value:+.3f}', va='center', fontsize=8)
fig.savefig(
    FIGURES_DIR / 'perdida_f1_macro_intervalos.png',
    dpi=300, bbox_inches='tight', pad_inches=0.08, facecolor='white'
)
plt.show()

# Figura 2. Curvas por familia y nivel.
curve_data = metrics.copy()
baseline_row = curve_data[curve_data['id'] == 'clean'].iloc[0]
families = conditions.query("id != 'clean'")['family'].drop_duplicates().tolist()
fig, axes = plt.subplots(2, 4, figsize=(12, 6.5), constrained_layout=True, sharey=True)
for ax, family in zip(axes.flat, families):
    family_rows = curve_data[curve_data['family'] == family].sort_values('level')
    levels = np.r_[0, family_rows['level'].to_numpy()]
    values = np.r_[baseline_row['macro_f1'], family_rows['macro_f1'].to_numpy()]
    ax.plot(levels, values, marker='o', linewidth=2.2, color='#1565c0')
    ax.set_title(family, fontweight='semibold', fontsize=10)
    ax.set_xticks([0, 1, 2], ['Original', 'Moderado', 'Severo'], rotation=20)
    ax.set_ylim(0, 1.03)
    ax.grid(alpha=0.25)
for ax in axes.flat[len(families):]:
    ax.axis('off')
axes[0, 0].set_ylabel('F1 macro')
axes[1, 0].set_ylabel('F1 macro')
fig.suptitle('Degradación del desempeño por intensidad', fontsize=15, fontweight='bold')
fig.savefig(
    FIGURES_DIR / 'curvas_robustez_por_familia.png',
    dpi=300, bbox_inches='tight', pad_inches=0.08, facecolor='white'
)
plt.show()


In [ ]:
# Figura 3. Ejemplo visual de las perturbaciones severas.
representative = clean_predictions.sort_values('confidence').iloc[len(clean_predictions) // 2]
representative_id = representative.name
representative_index = test_manifest.index[test_manifest['image_id'] == representative_id][0]
example_condition_ids = [
    'clean', 'dark_severe', 'bright_severe', 'contrast_severe',
    'blur_severe', 'noise_severe', 'rotation_severe', 'jpeg_severe',
]

fig, axes = plt.subplots(2, 4, figsize=(11, 6.6), constrained_layout=True)
for ax, condition_id in zip(axes.flat, example_condition_ids):
    condition = conditions.set_index('id').loc[condition_id]
    image_path = paths_by_condition[condition_id][representative_index]
    with Image.open(image_path) as image:
        ax.imshow(ImageOps.exif_transpose(image).convert('RGB'))
    ax.set_title(condition['label'], fontsize=10, fontweight='semibold')
    ax.axis('off')
fig.suptitle(
    f"Perturbaciones simuladas · Clase real: {SPANISH[representative['true_class']].upper()}",
    fontsize=14, fontweight='bold'
)
fig.savefig(
    FIGURES_DIR / 'ejemplo_perturbaciones_controladas.png',
    dpi=300, bbox_inches='tight', pad_inches=0.08, facecolor='white'
)
plt.show()

# Figura 4. Matriz de confusión normalizada de la condición con menor F1 macro.
worst = metrics.query("id != 'clean'").sort_values('macro_f1').iloc[0]
worst_frame = predictions[predictions['condition_id'] == worst['id']]
cm = confusion_matrix(
    worst_frame['true_class'], worst_frame['predicted_class'],
    labels=CLASSES, normalize='true'
)
labels_es = [SPANISH[label].capitalize() for label in CLASSES]
fig, ax = plt.subplots(figsize=(7.4, 6.2), constrained_layout=True)
sns.heatmap(
    cm, annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1,
    xticklabels=labels_es, yticklabels=labels_es, linewidths=0.5,
    cbar_kws={'label': 'Proporción por clase real'}, ax=ax
)
ax.set_xlabel('Clase predicha')
ax.set_ylabel('Clase real')
ax.set_title(
    f"Condición más exigente: {worst['label']} · F1 macro = {worst['macro_f1']:.3f}",
    fontweight='bold'
)
fig.savefig(
    FIGURES_DIR / 'matriz_confusion_peor_condicion.png',
    dpi=300, bbox_inches='tight', pad_inches=0.08, facecolor='white'
)
plt.show()


## 7. Síntesis automática y archivos de salida

El texto generado resume los resultados observados y puede servir como borrador para el capítulo del libro. Debe revisarse junto con las figuras antes de integrarlo al manuscrito.


In [ ]:
clean_row = metrics.loc[metrics['id'] == 'clean'].iloc[0]
worst_row = metrics.query("id != 'clean'").sort_values('macro_f1').iloc[0]
best_perturbed_row = metrics.query("id != 'clean'").sort_values('macro_f1', ascending=False).iloc[0]

worst_class_frame = per_class[per_class['condition_id'] == worst_row['id']].copy()
clean_recall = (
    per_class[per_class['condition_id'] == 'clean']
    .set_index('class')['recall']
)
worst_class_frame['recall_drop'] = worst_class_frame.apply(
    lambda row: clean_recall[row['class']] - row['recall'], axis=1
)
most_affected_class = worst_class_frame.sort_values('recall_drop', ascending=False).iloc[0]

interpretation = f'''# Síntesis de robustez

En las {len(test_manifest)} imágenes del conjunto de prueba, el modelo obtuvo un F1 macro de {clean_row['macro_f1']:.3f} y una exactitud balanceada de {clean_row['balanced_accuracy']:.3f} en la condición original. La perturbación con menor desempeño fue **{worst_row['label']}**, con un F1 macro de {worst_row['macro_f1']:.3f}, una pérdida absoluta de {worst_row['macro_f1_drop']:.3f} y una tasa de cambio de etiqueta de {worst_row['flip_rate']:.1%}. La clase con mayor reducción de sensibilidad bajo esa condición fue **{most_affected_class['class_es']}**, con una disminución de recall de {most_affected_class['recall_drop']:.3f}.

La condición perturbada con mayor F1 macro fue **{best_perturbed_row['label']}** ({best_perturbed_row['macro_f1']:.3f}). Estos resultados describen sensibilidad a alteraciones simuladas sobre las mismas imágenes y no constituyen validación externa. Antes de un uso agronómico se requieren fotografías nuevas, obtenidas con otros dispositivos, fondos, lotes y condiciones ambientales.
'''
(RUN_BOOK / 'sintesis_resultados_para_libro.md').write_text(interpretation, encoding='utf-8')
print(interpretation)

elapsed_total = time.perf_counter() - generation_start
metadata = {
    'run_id': RUN_ID,
    'model': 'YOLO11m-cls',
    'seed_model': 2026,
    'seed_perturbations_and_bootstrap': SEED,
    'image_size': IMGSZ,
    'test_images': len(test_manifest),
    'conditions_including_clean': len(conditions),
    'total_predictions': len(predictions),
    'bootstrap_repetitions': BOOTSTRAP_REPETITIONS,
    'device': DEVICE_NAME,
    'elapsed_seconds_from_perturbation_generation': round(elapsed_total, 3),
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
    'scope_warning': 'Perturbaciones simuladas; no sustituyen validación externa de campo.',
}
(RUN_RESULTS / 'metadatos_nb07.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)

print('NB07 FINALIZADO CORRECTAMENTE')
print('Resultados:', RUN_RESULTS)
print('Figuras y tablas para el libro:', RUN_BOOK)


## Interpretación responsable

- Una disminución de F1 macro muestra sensibilidad del modelo a una alteración concreta.
- Una tasa alta de cambio de etiqueta indica inestabilidad, aunque algunos cambios pueden corregir errores originales.
- Un valor *p* significativo debe acompañarse siempre del tamaño del efecto y su intervalo de confianza.
- Las 70 imágenes transformadas no son observaciones nuevas; son versiones pareadas de las mismas fotografías.
- No debe ajustarse el modelo después de observar estos resultados y volver a presentar el mismo conjunto como una prueba independiente.
- La siguiente etapa recomendable es una validación externa con imágenes nuevas de campo.
